# Cross-Event Regression Test — Single Station

Idea: prendere **una** stazione, fittare un regressore **MISO-lag (Ridge)** su un evento 
(finestra −15 / +5 giorni attorno alla data del picco) e poi **testarlo su un altro evento** 
(stessa logica di finestra) per vedere quanto la stessa struttura dinamica (IRF) 
impulse-response si trasferisce da una tempesta all'altra.

**Modello:**
$$\varepsilon(t) \;=\; \sum_{i=1}^{4} \sum_{k=0}^{L} \beta_{i,k}\, x_i(t-k) \;+\; \eta(t)$$

dove $\varepsilon(t)$ è l'errore model−obs, $x_i$ = SLP, T2m, u10, v10, $L$ = max lag.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.stattools import durbin_watson

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

# ── 1. Load the station ────────────────────────────────────────────────
DATA_DIR = Path("/Users/nicolocaron/Desktop/MASTER PROJECT/"
                "DATASET COMPLETI 6 STAZIONI INIZIALI/per_station")
STATION_FILE = "station_30336_Kobenhavn.parquet"   # ← change here to use another station

df = pd.read_parquet(DATA_DIR / STATION_FILE)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)
df["error_m"] = df["forcoast_p82_m"] - df["tg_obs_m"]   # ε(t) = model − obs

STATION_NAME = df["station_name"].iloc[0]
print(f"Stazione: {STATION_NAME}")
print(f"Periodo disponibile: {df['time'].min()}  →  {df['time'].max()}")
print(f"N ore totali: {len(df):,}")

# Mostra i top 10 picchi osservati (per scegliere le date input)
top = df.nlargest(10, "tg_obs_m")[["time", "tg_obs_m", "forcoast_p82_m", "error_m"]]
print("\nTop 10 picchi TG osservati:")
print(top.to_string(index=False))

## Parametri globali

In [ ]:
# Finestra: 15 giorni prima del picco, 5 giorni dopo
DAYS_BEFORE = 15
DAYS_AFTER  = 5

# Lag massimo (ore) per MISO
L = 72     # → memoria di 3 giorni

FEATURES = ["SLP", "t2m", "u10", "v10"]


def slice_event(df: pd.DataFrame, peak_date: str) -> pd.DataFrame:
    """Return the event window −DAYS_BEFORE / +DAYS_AFTER around `peak_date`."""
    t0 = pd.Timestamp(peak_date)
    start = t0 - pd.Timedelta(days=DAYS_BEFORE)
    end   = t0 + pd.Timedelta(days=DAYS_AFTER)
    sub = df[(df["time"] >= start) & (df["time"] <= end)].copy()
    sub = sub.dropna(subset=FEATURES + ["error_m"]).reset_index(drop=True)
    return sub, t0, start, end


def build_lag_matrix(sub: pd.DataFrame, features: list, L: int):
    """Build the design matrix X with lag 0..L for each feature."""
    cols = {}
    for feat in features:
        s = sub[feat].values
        for k in range(L + 1):
            cols[f"{feat}_lag{k}"] = np.concatenate([np.full(k, np.nan), s[:-k] if k > 0 else s])
    X = pd.DataFrame(cols)
    y = sub["error_m"].values
    t = sub["time"].values
    # Drop initial rows with NaN from lags
    mask = X.notna().all(axis=1).values
    return X.values[mask], y[mask], t[mask]

print(f"Finestra evento: −{DAYS_BEFORE} / +{DAYS_AFTER} giorni  →  {(DAYS_BEFORE+DAYS_AFTER)*24} ore")
print(f"Lag massimo:      L = {L} h")
print(f"Features:         {FEATURES}")

## Cella 1 — TRAINING

Inserisci la data del **picco su cui allenare**. Il modello MISO-lag + Ridge viene fittato 
sulla finestra −15 / +5 giorni, con standardizzazione delle feature.

In [ ]:
# ────────────────────────────────────────────────────────────
#  👉 INSERISCI QUI LA DATA DEL PICCO DI TRAINING (YYYY-MM-DD)
# ────────────────────────────────────────────────────────────
TRAIN_PEAK = "2017-10-29"

# 1. Slice della finestra evento
sub_tr, t0_tr, start_tr, end_tr = slice_event(df, TRAIN_PEAK)
print(f"TRAIN window: {start_tr}  →  {end_tr}   (picco nominale {t0_tr})")
print(f"Righe: {len(sub_tr)}")

# 2. Lag design matrix
X_tr, y_tr, t_tr = build_lag_matrix(sub_tr, FEATURES, L)
print(f"X_tr shape: {X_tr.shape}    y_tr shape: {y_tr.shape}")

# 3. Standardize + RidgeCV
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)

model = RidgeCV(alphas=np.logspace(-3, 3, 25), cv=5).fit(X_tr_s, y_tr)
alpha_opt = model.alpha_

# 4. Metriche in-sample
y_hat = model.predict(X_tr_s)
res   = y_tr - y_hat
r2_tr   = r2_score(y_tr, y_hat)
rmse_tr = np.sqrt(mean_squared_error(y_tr, y_hat))
dw_tr   = durbin_watson(res)

print(f"\n🔧 Ridge α*       = {alpha_opt:.3g}")
print(f"📊 In-sample R²    = {r2_tr:+.4f}")
print(f"📊 In-sample RMSE  = {rmse_tr:.4f} m")
print(f"📊 Durbin-Watson   = {dw_tr:.2f}")

# 5. Salvo tutto in un dict per la cella di TEST
trained = {
    "model":    model,
    "scaler":   scaler,
    "features": FEATURES,
    "L":        L,
    "peak":     TRAIN_PEAK,
    "alpha":    alpha_opt,
    "r2":       r2_tr,
    "rmse":     rmse_tr,
}

# 6. Plot training: ε osservato vs fitted
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                        gridspec_kw={"height_ratios": [1, 1]})
ax[0].plot(sub_tr["time"], sub_tr["tg_obs_m"],       color="#2980B9", lw=1.0, label="TG obs")
ax[0].plot(sub_tr["time"], sub_tr["forcoast_p82_m"], color="#E74C3C", lw=1.0, label="Model (HDM)")
ax[0].axvline(t0_tr, ls=":", color="grey")
ax[0].set_ylabel("Sea level [m]")
ax[0].legend(fontsize=8, loc="upper left")
ax[0].set_title(f"TRAIN — {STATION_NAME}  ·  peak {TRAIN_PEAK}  (−{DAYS_BEFORE}/+{DAYS_AFTER} d)",
                fontweight="bold")

ax[1].plot(t_tr, y_tr,   color="#8E44AD", lw=0.9, label="ε observed")
ax[1].plot(t_tr, y_hat,  color="#27AE60", lw=1.1, label=f"ε fitted  (R²={r2_tr:+.3f})")
ax[1].axhline(0, color="grey", lw=0.6)
ax[1].axvline(t0_tr, ls=":", color="grey")
ax[1].set_ylabel("ε = model − obs [m]")
ax[1].set_xlabel("time")
ax[1].legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## Cella 2 — TEST (cross-event)

Inserisci una **data di picco diversa**. Il modello allenato sopra viene applicato 
alla nuova finestra −15 / +5 giorni **senza re-fit**. Vediamo se i coefficienti
impulse-response sono generalizzabili ad altre tempeste.

In [ ]:
# ────────────────────────────────────────────────────────────
#  👉 INSERISCI QUI LA DATA DEL PICCO DI TEST (YYYY-MM-DD)
# ────────────────────────────────────────────────────────────
TEST_PEAK = "2019-01-02"

assert "trained" in globals(), "Esegui prima la cella di TRAINING!"

# 1. Slice finestra test
sub_te, t0_te, start_te, end_te = slice_event(df, TEST_PEAK)
print(f"TEST window : {start_te}  →  {end_te}   (picco nominale {t0_te})")
print(f"Righe: {len(sub_te)}")

# 2. Costruisci X con la stessa struttura di lag
X_te, y_te, t_te = build_lag_matrix(sub_te, trained["features"], trained["L"])

# 3. Applica scaler e modello TRAINATI (niente re-fit!)
X_te_s = trained["scaler"].transform(X_te)
y_pred = trained["model"].predict(X_te_s)
res_te = y_te - y_pred

# 4. Metriche out-of-sample
r2_te    = r2_score(y_te, y_pred)
rmse_te  = np.sqrt(mean_squared_error(y_te, y_pred))
bias_te  = float(np.mean(res_te))
dw_te    = durbin_watson(res_te)
# Residual variance reduction vs "no correction" (baseline = ε stesso)
ss_res   = np.sum(res_te**2)
ss_tot   = np.sum((y_te - y_te.mean())**2)
skill    = 1 - ss_res / ss_tot   # coincide con R², ma lo stampiamo esplicitamente

print("\n┏━━━ TEST (out-of-sample) ━━━━━━━━━━━━━━━━━━━━┓")
print(f"  Train peak: {trained['peak']}")
print(f"  Test  peak: {TEST_PEAK}")
print(f"  R²        = {r2_te:+.4f}   (train: {trained['r2']:+.4f})")
print(f"  RMSE      = {rmse_te:.4f} m (train: {trained['rmse']:.4f} m)")
print(f"  Bias      = {bias_te:+.4f} m")
print(f"  Skill     = {skill:+.4f}")
print(f"  DW        = {dw_te:.2f}")
print("┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛")

# 5. Plot: ε osservato vs predetto sulla finestra di test
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True,
                        gridspec_kw={"height_ratios": [1, 1, 0.6]})

ax[0].plot(sub_te["time"], sub_te["tg_obs_m"],       color="#2980B9", lw=1.0, label="TG obs")
ax[0].plot(sub_te["time"], sub_te["forcoast_p82_m"], color="#E74C3C", lw=1.0, label="Model (HDM)")
ax[0].axvline(t0_te, ls=":", color="grey")
ax[0].set_ylabel("Sea level [m]")
ax[0].legend(fontsize=8, loc="upper left")
ax[0].set_title(
    f"TEST — {STATION_NAME}  ·  peak {TEST_PEAK}  "
    f"(modello addestrato su {trained['peak']})",
    fontweight="bold")

ax[1].plot(t_te, y_te,   color="#8E44AD", lw=0.9, label="ε observed")
ax[1].plot(t_te, y_pred, color="#27AE60", lw=1.1,
            label=f"ε predicted  (R²={r2_te:+.3f}, RMSE={rmse_te:.3f} m)")
ax[1].axhline(0, color="grey", lw=0.6)
ax[1].axvline(t0_te, ls=":", color="grey")
ax[1].set_ylabel("ε [m]")
ax[1].legend(fontsize=8, loc="upper left")

# Corrected model = HDM − ε̂ (ε̂ = previsione dell'errore)
hdm_vals  = sub_te.set_index("time").loc[pd.to_datetime(t_te), "forcoast_p82_m"].values
tg_vals   = sub_te.set_index("time").loc[pd.to_datetime(t_te), "tg_obs_m"].values
hdm_corr  = hdm_vals - y_pred
rmse_raw  = np.sqrt(np.mean((hdm_vals - tg_vals)**2))
rmse_corr = np.sqrt(np.mean((hdm_corr - tg_vals)**2))

ax[2].plot(t_te, hdm_vals - tg_vals, color="#E74C3C", lw=0.9,
            label=f"raw error   RMSE={rmse_raw:.3f} m")
ax[2].plot(t_te, hdm_corr - tg_vals, color="#27AE60", lw=1.1,
            label=f"after corr. RMSE={rmse_corr:.3f} m")
ax[2].axhline(0, color="grey", lw=0.6)
ax[2].set_ylabel("model − obs [m]")
ax[2].set_xlabel("time")
ax[2].legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

pct_gain = 100 * (rmse_raw - rmse_corr) / rmse_raw
print(f"📉 RMSE riduzione dopo correzione: {rmse_raw:.4f} → {rmse_corr:.4f} m  ({pct_gain:+.1f}%)")

## (opzionale) Impulse Response Functions del modello allenato

I coefficienti $\beta_{i,k}$ vengono riportati al loro segno fisico 
(moltiplicando per $\sigma_{x_i}$ della standardizzazione). 
Se il modello è fisicamente sensato, le IRF dovrebbero essere simili tra diversi eventi.

In [ ]:
coefs = trained["model"].coef_     # lunghezza = n_feat * (L+1)
n_feat = len(trained["features"])
Lp1 = trained["L"] + 1
beta = coefs.reshape(n_feat, Lp1)  # [features × lags]

fig, axes = plt.subplots(1, n_feat, figsize=(14, 3), sharey=True)
colors = ["#E74C3C", "#3498DB", "#2ECC71", "#F39C12"]
for i, (feat, ax) in enumerate(zip(trained["features"], axes)):
    ax.stem(range(Lp1), beta[i], linefmt=colors[i], markerfmt="o", basefmt="grey")
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_title(f"IRF — {feat}", fontweight="bold")
    ax.set_xlabel("lag [h]")
axes[0].set_ylabel(r"$\beta$  [m / σ]")
plt.suptitle(f"Impulse Response — train peak {trained['peak']}  (L={trained['L']} h)",
             fontweight="bold")
plt.tight_layout()
plt.show()